# NeuralRetail AI - Exploratory Data Analysis (EDA)
This notebook performs exploratory data analysis on the Online Retail II dataset, analyzing sales trends, product performance, country-wise revenue, and customer behavior.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load Preprocessed Data

In [ ]:
# Set base directory path
BASE_DIR = Path("..")
cleaned_path = BASE_DIR / "data" / "processed" / "cleaned_retail.csv"

df = pd.read_csv(cleaned_path)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Visualizations

### 1. Revenue Distribution
Analyze the distribution of order line item revenues (`TotalPrice`).

In [ ]:
fig_rev_dist = px.histogram(df[df["TotalPrice"] < 100], x="TotalPrice", nbins=50, 
                            title="Distribution of Transaction Line Item Revenues (Clamped < $100)",
                            labels={"TotalPrice": "Line Item Revenue ($)"},
                            color_discrete_sequence=["#6366F1"])
fig_rev_dist.show()

### 2. Monthly Revenue Trend
Track total revenue per month over time.

In [ ]:
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M").astype(str)
monthly_rev = df.groupby("YearMonth")["TotalPrice"].sum().reset_index()

fig_monthly = px.line(monthly_rev, x="YearMonth", y="TotalPrice", markers=True,
                      title="Monthly Revenue Trend (2009 - 2011)",
                      labels={"TotalPrice": "Total Sales ($)", "YearMonth": "Month"},
                      color_discrete_sequence=["#10B981"])
fig_monthly.show()

### 3. Daily Revenue Trend
Track total sales revenue per day.

In [ ]:
df["Date"] = df["InvoiceDate"].dt.date
daily_rev = df.groupby("Date")["TotalPrice"].sum().reset_index()

fig_daily = px.line(daily_rev, x="Date", y="TotalPrice",
                    title="Daily Revenue Trend",
                    labels={"TotalPrice": "Daily Sales ($)"},
                    color_discrete_sequence=["#8B5CF6"])
fig_daily.show()

### 4. Country Wise Revenue
Compare total revenue generated across different countries.

In [ ]:
country_rev = df.groupby("Country")["TotalPrice"].sum().reset_index().sort_values(by="TotalPrice", ascending=False)
top_countries = country_rev.head(10)

fig_country = px.bar(top_countries, x="TotalPrice", y="Country", orientation="h",
                     title="Top 10 Countries by Total Revenue",
                     labels={"TotalPrice": "Revenue ($)"},
                     color="TotalPrice", color_continuous_scale="Viridis")
fig_country.update_layout(yaxis=dict(autorange="reversed"))
fig_country.show()

### 5. Top Products
Analyze products generating the highest revenue.

In [ ]:
top_products = df.groupby("Description")["TotalPrice"].sum().reset_index().sort_values(by="TotalPrice", ascending=False).head(10)

fig_prod = px.bar(top_products, x="TotalPrice", y="Description", orientation="h",
                  title="Top 10 Products by Total Revenue",
                  labels={"TotalPrice": "Revenue ($)", "Description": "Product Description"},
                  color="TotalPrice", color_continuous_scale="Cividis")
fig_prod.update_layout(yaxis=dict(autorange="reversed"))
fig_prod.show()

### 6. Customer Purchase Distribution
Understand how much customers typically spend in their lifetime.

In [ ]:
cust_spend = df.groupby("Customer ID")["TotalPrice"].sum().reset_index()

fig_cust_spend = px.box(cust_spend[cust_spend["TotalPrice"] < 5000], x="TotalPrice",
                        title="Distribution of Lifetime Spending per Customer (Clamped < $5,000)",
                        labels={"TotalPrice": "Customer Lifetime Spend ($)"},
                        color_discrete_sequence=["#EF4444"])
fig_cust_spend.show()

### 7. Order Distribution
Distribution of the total items in each order.

In [ ]:
order_qty = df.groupby("Invoice")["Quantity"].sum().reset_index()

fig_order_qty = px.histogram(order_qty[order_qty["Quantity"] < 1000], x="Quantity", nbins=50,
                             title="Distribution of Order Quantities (Clamped < 1000 Units)",
                             labels={"Quantity": "Total Quantity in Order"},
                             color_discrete_sequence=["#F59E0B"])
fig_order_qty.show()

### 8. Revenue by Country (Percentage share)
Share of sales revenue generated outside the UK vs UK.

In [ ]:
country_shares = country_rev.copy()
uk_revenue = country_shares[country_shares["Country"] == "United Kingdom"]["TotalPrice"].sum()
non_uk_revenue = country_shares[country_shares["Country"] != "United Kingdom"]["TotalPrice"].sum()

fig_pie = px.pie(names=["United Kingdom", "Rest of the World"], values=[uk_revenue, non_uk_revenue],
                 title="Revenue Share: United Kingdom vs Rest of World",
                 color_discrete_sequence=["#3B82F6", "#10B981"])
fig_pie.show()

### 9. Revenue by Product (Top 20 Product Categories)

In [ ]:
top_20_products = df.groupby("Description")["TotalPrice"].sum().reset_index().sort_values(by="TotalPrice", ascending=False).head(20)
fig_treemap = px.treemap(top_20_products, path=["Description"], values="TotalPrice",
                         title="Top 20 Revenue Generating Products Map",
                         color="TotalPrice", color_continuous_scale="RdBu")
fig_treemap.show()

### 10. Customer Purchase Frequency
Distribution of the number of orders made by each customer.

In [ ]:
cust_freq = df.groupby("Customer ID")["Invoice"].nunique().reset_index()

fig_freq = px.histogram(cust_freq[cust_freq["Invoice"] < 30], x="Invoice", nbins=30,
                        title="Distribution of Purchase Frequency per Customer (Clamped < 30 Orders)",
                        labels={"Invoice": "Number of Orders"},
                        color_discrete_sequence=["#EC4899"])
fig_freq.show()